# 5. Writer-ID feature extractor (SigNet-style), for models D and E

Trains EfficientNet-B2 to recognise **which of the 36 training writers** wrote a genuine signature. The resulting
features describe *how a person signs*, which is what comparing a questioned signature with specimens needs.

Validation writers are deliberately left out: they pick the SVM decision threshold in `wd_svm.ipynb`, and that only
transfers to test writers if the feature extractor has never seen them either. Test writers are never seen.

* `LAM = 0` → **SigNet** (writer-ID head only, genuine signatures only) → `runs/writerid_fold<k>/best.pt`
* `LAM > 0` → **SigNet-F**: a second head also learns genuine vs skilled forgery on the same writers,
  loss = CE_writer(genuine) + LAM · BCE_forged(all) → `runs/writeridF_fold<k>/best.pt`

Saves the backbone weights of the last epoch (no model selection, so test writers stay untouched).
Existing checkpoints are kept unless `RETRAIN = True` (≈ 7 min per fold for SigNet, 13 min for SigNet-F).

In [1]:
LAM = 0.0                # 0 = SigNet, 1.0 = SigNet-F
FOLDS = [0, 1, 2, 3, 4]
EPOCHS = 20
BATCH = 16
LR = 1e-4
WEIGHT_DECAY = 0.05
RETRAIN = False

In [2]:
%run ./common.ipynb

In [3]:
import time


class WriterNet(nn.Module):
    def __init__(self, n_writers):
        super().__init__()
        b = build_model()
        self.features, self.pool = b.features, b.avgpool
        self.drop = nn.Dropout(0.5)
        self.writer_head = nn.Linear(1408, n_writers)
        self.forg_head = nn.Linear(1408, 1)

    def forward(self, x):
        f = self.drop(torch.flatten(self.pool(self.features(x)), 1))
        return self.writer_head(f), self.forg_head(f).squeeze(1)


def train_writerid_fold(fold):
    name = f"writerid{'F' if LAM > 0 else ''}_fold{fold}"
    out = os.path.join(RUNS, name)
    os.makedirs(out, exist_ok=True)
    torch.manual_seed(42)
    device = DEVICE

    meta, tr, va, te = writer_split(fold=fold)
    idx = tr
    if LAM == 0:
        idx = idx[meta.label.values[idx] == 0]                 # writer ID uses genuine only
    writers = sorted(set(meta.writer.values[idx]))
    wmap = {w: i for i, w in enumerate(writers)}
    wid = np.array([wmap[w] for w in meta.writer.values[idx]])
    forged = meta.label.values[idx]
    print(f"[{name}] {len(idx)} images from {len(writers)} writers; test writers {sorted(set(meta.writer[te]))}")

    G = load_gray("binary")
    images = SignatureSet(np.asarray(G[idx]), forged, True)

    class WriterItems(Dataset):                                # (image, writer id, forged?)
        def __len__(self):
            return len(idx)

        def __getitem__(self, i):
            return images[i][0], int(wid[i]), np.float32(forged[i])

    dl = ThreadLoader(WriterItems(), batch_size=BATCH, shuffle=True, drop_last=True)

    model = WriterNet(len(writers)).to(device)
    heads = list(model.writer_head.parameters()) + list(model.forg_head.parameters())
    opt = torch.optim.AdamW([{"params": model.features.parameters(), "lr": LR},
                             {"params": heads, "lr": LR * 10}], weight_decay=WEIGHT_DECAY)
    sched = warmup_cosine(opt, len(dl), EPOCHS * len(dl))
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    ce, bce = nn.CrossEntropyLoss(label_smoothing=0.1), nn.BCEWithLogitsLoss()

    for ep in range(1, EPOCHS + 1):
        model.train()
        t0, run, correct, seen = time.time(), 0.0, 0, 0
        for xb, wb, fb in dl:
            xb, wb, fb = xb.to(device), wb.to(device), fb.to(device)
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                wl, fl = model(xb)
                gen = fb == 0
                loss = ce(wl[gen], wb[gen]) if gen.any() else wl.sum() * 0
                if LAM > 0:
                    loss = loss + LAM * bce(fl, fb * 0.9 + 0.05)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run += loss.item()
            correct += (wl[gen].argmax(1) == wb[gen]).sum().item()
            seen += gen.sum().item()
        print(f"ep {ep:2d}  loss {run / len(dl):.4f}  train writer-ID acc {correct / max(seen, 1):.3f}  "
              f"({time.time() - t0:.0f}s)", flush=True)

    torch.save({"features": model.features.state_dict()}, os.path.join(out, "best.pt"))
    print("saved", os.path.join(out, "best.pt"))

In [4]:
for fold in FOLDS:
    ckpt = os.path.join(RUNS, f"writerid{'F' if LAM > 0 else ''}_fold{fold}", "best.pt")
    if os.path.exists(ckpt) and not RETRAIN:
        print(f"fold {fold}: checkpoint exists ({ckpt})")
    else:
        train_writerid_fold(fold)

fold 0: checkpoint exists (C:\Sankar Sriram\Bank Details\DIP\DIP_CASE_STUDY\runs\writerid_fold0\best.pt)
fold 1: checkpoint exists (C:\Sankar Sriram\Bank Details\DIP\DIP_CASE_STUDY\runs\writerid_fold1\best.pt)
fold 2: checkpoint exists (C:\Sankar Sriram\Bank Details\DIP\DIP_CASE_STUDY\runs\writerid_fold2\best.pt)
fold 3: checkpoint exists (C:\Sankar Sriram\Bank Details\DIP\DIP_CASE_STUDY\runs\writerid_fold3\best.pt)
fold 4: checkpoint exists (C:\Sankar Sriram\Bank Details\DIP\DIP_CASE_STUDY\runs\writerid_fold4\best.pt)
